# Module 5: Ray on SLURM

**DSC 232R - Big Data Analysis Using Spark**

This notebook covers deploying Ray on HPC clusters:
1. Multi-node Ray cluster architecture
2. SLURM scripts for Ray
3. Connecting to Ray clusters
4. Scaling considerations

**Note**: Most commands in this notebook are for reference - they require an actual SLURM cluster to execute.

## Key Takeaways

- **Ray clusters** can span multiple SLURM nodes for massive scale
- **Head + Worker** architecture maps to SLURM node allocation
- **Singularity containers** ensure consistent environments
- **Multi-node** enables processing datasets larger than single-node memory

In [ ]:
!pip install ray

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 11.1 MB/s eta 0:00:00


In [ ]:
import ray
import numpy as np
import pandas as pd
import os

print(f"Ray version: {ray.__version__}")

Ray version: 2.55.1


---

## 1. Multi-Node Architecture

### Single vs Multi-Node Comparison

In [ ]:
# Node configuration on SDSC Expanse
node_specs = {
    "CPU Cores": 128,
    "RAM (GB)": 256,
    "Local SSD (GB)": 1000,
}

# Compare single vs multi-node
configs = pd.DataFrame({
    "Metric": ["Total Cores", "Total RAM (GB)", "Max Data In Memory (GB)"],
    "1 Node": [128, 256, 180],
    "4 Nodes": [512, 1024, 720],
    "8 Nodes": [1024, 2048, 1440],
    "16 Nodes": [2048, 4096, 2880],
})

print("SDSC Expanse - Resource Scaling")
print("=" * 60)
print(configs.to_string(index=False))
print("\n* Max Data In Memory assumes ~70% of RAM for Ray object store")

SDSC Expanse - Resource Scaling
                 Metric  1 Node  4 Nodes  8 Nodes  16 Nodes
            Total Cores     128      512     1024      2048
         Total RAM (GB)     256     1024     2048      4096
Max Data In Memory (GB)     180      720     1440      2880

* Max Data In Memory assumes ~70% of RAM for Ray object store


### When to Use Multi-Node

In [ ]:
scenarios = pd.DataFrame({
    "Dataset Size": ["< 100 GB", "100-500 GB", "500 GB - 1 TB", "> 1 TB"],
    "Recommended Nodes": ["1", "2-4", "4-8", "8+"],
    "Partition": ["shared", "compute", "compute", "compute"],
    "Notes": [
        "Single node is sufficient",
        "Multi-node for faster processing",
        "Required for memory capacity",
        "Consider data partitioning"
    ]
})

print("Multi-Node Scaling Guide")
print("=" * 70)
print(scenarios.to_string(index=False))

Multi-Node Scaling Guide
 Dataset Size Recommended Nodes Partition                            Notes
     < 100 GB                 1    shared        Single node is sufficient
   100-500 GB               2-4   compute Multi-node for faster processing
500 GB - 1 TB               4-8   compute     Required for memory capacity
       > 1 TB                8+   compute       Consider data partitioning


---

## 2. SLURM Script Components

### Multi-Node SLURM Script Structure

In [ ]:
slurm_script_template = '''
#!/bin/bash
#SBATCH --job-name=ray_cluster
#SBATCH --partition=compute        # Use compute for multi-node
#SBATCH --nodes=4                  # Number of nodes
#SBATCH --ntasks-per-node=1        # One task per node
#SBATCH --cpus-per-task=64         # CPUs per node
#SBATCH --mem=200G                 # Memory per node
#SBATCH --time=04:00:00            # Time limit
#SBATCH --account=uci150           # Allocation

# Load container
module load singularitypro
CONTAINER=/path/to/ray_spark_dsc232r.sif

# Get head node
HEAD_NODE=$(scontrol show hostnames $SLURM_NODELIST | head -n 1)
HEAD_IP=$(srun --nodes=1 --ntasks=1 -w $HEAD_NODE hostname -i)

# Start head node
srun --nodes=1 --ntasks=1 -w $HEAD_NODE \\
    singularity exec $CONTAINER \\
    ray start --head --port=6379 --num-cpus=64 --block &

sleep 10

# Start worker nodes
for NODE in $(scontrol show hostnames $SLURM_NODELIST | tail -n +2); do
    srun --nodes=1 --ntasks=1 -w $NODE \\
        singularity exec $CONTAINER \\
        ray start --address=$HEAD_IP:6379 --num-cpus=64 --block &
done

sleep 20

# Run application
singularity exec $CONTAINER python my_script.py --ray-address=$HEAD_IP:6379
'''

print("Multi-Node SLURM Script Template")
print("=" * 50)
print(slurm_script_template)

Multi-Node SLURM Script Template

#!/bin/bash
#SBATCH --job-name=ray_cluster
#SBATCH --partition=compute        # Use compute for multi-node
#SBATCH --nodes=4                  # Number of nodes
#SBATCH --ntasks-per-node=1        # One task per node
#SBATCH --cpus-per-task=64         # CPUs per node
#SBATCH --mem=200G                 # Memory per node
#SBATCH --time=04:00:00            # Time limit
#SBATCH --account=uci150           # Allocation

# Load container
module load singularitypro
CONTAINER=/path/to/ray_spark_dsc232r.sif

# Get head node
HEAD_NODE=$(scontrol show hostnames $SLURM_NODELIST | head -n 1)
HEAD_IP=$(srun --nodes=1 --ntasks=1 -w $HEAD_NODE hostname -i)

# Start head node
srun --nodes=1 --ntasks=1 -w $HEAD_NODE \
    singularity exec $CONTAINER \
    ray start --head --port=6379 --num-cpus=64 --block &

sleep 10

# Start worker nodes
for NODE in $(scontrol show hostnames $SLURM_NODELIST | tail -n +2); do
    srun --nodes=1 --ntasks=1 -w $NODE \
        singularity exe

### Key SLURM Variables

In [ ]:
slurm_vars = pd.DataFrame({
    "Variable": [
        "$SLURM_NODELIST",
        "$SLURM_NNODES",
        "$SLURM_CPUS_PER_TASK",
        "$SLURM_JOB_ID",
        "$SLURM_MEM_PER_NODE"
    ],
    "Description": [
        "List of allocated node names",
        "Number of nodes in allocation",
        "CPUs allocated per task",
        "Unique job identifier",
        "Memory per node"
    ],
    "Example Value": [
        "exp-15-[01-04]",
        "4",
        "64",
        "12345678",
        "200G"
    ]
})

print("Key SLURM Environment Variables")
print("=" * 70)
print(slurm_vars.to_string(index=False))

Key SLURM Environment Variables
            Variable                   Description  Example Value
     $SLURM_NODELIST  List of allocated node names exp-15-[01-04]
       $SLURM_NNODES Number of nodes in allocation              4
$SLURM_CPUS_PER_TASK       CPUs allocated per task             64
       $SLURM_JOB_ID         Unique job identifier       12345678
 $SLURM_MEM_PER_NODE               Memory per node           200G


---

## 3. Connecting to Ray Cluster

### Python Connection Code

In [ ]:
# Template for connecting to SLURM-launched Ray cluster

connection_code = '''
import ray
import os
import argparse

def connect_to_ray_cluster():
    """
    Connect to Ray cluster started by SLURM.

    Usage:
        python script.py --ray-address=10.0.0.1:6379
    """
    parser = argparse.ArgumentParser()
    parser.add_argument('--ray-address', default='auto',
                        help='Ray head node address (ip:port)')
    args = parser.parse_args()

    # Connect to cluster
    ray.init(address=args.ray_address)

    # Verify connection
    print(f"Ray version: {ray.__version__}")
    print(f"Cluster resources: {ray.cluster_resources()}")

    return ray

if __name__ == "__main__":
    connect_to_ray_cluster()

    # Your code here...

    ray.shutdown()
'''

print("Ray Cluster Connection Template")
print("=" * 50)
print(connection_code)

Ray Cluster Connection Template

import ray
import os
import argparse

def connect_to_ray_cluster():
    """
    Connect to Ray cluster started by SLURM.
    
    Usage:
        python script.py --ray-address=10.0.0.1:6379
    """
    parser = argparse.ArgumentParser()
    parser.add_argument('--ray-address', default='auto',
                        help='Ray head node address (ip:port)')
    args = parser.parse_args()
    
    # Connect to cluster
    ray.init(address=args.ray_address)
    
    # Verify connection
    print(f"Ray version: {ray.__version__}")
    print(f"Cluster resources: {ray.cluster_resources()}")
    
    return ray

if __name__ == "__main__":
    connect_to_ray_cluster()
    
    # Your code here...
    
    ray.shutdown()



### Verifying Multi-Node Execution

In [ ]:
# Code to verify tasks are distributed across nodes

verification_code = '''
@ray.remote
def get_node_info():
    """Return information about the node running this task."""
    import socket
    import os
    return {
        "hostname": socket.gethostname(),
        "pid": os.getpid(),
        "ray_node_id": ray.get_runtime_context().get_node_id(),
    }

# Run on multiple workers
n_tasks = 100
futures = [get_node_info.remote() for _ in range(n_tasks)]
results = ray.get(futures)

# Count unique nodes
unique_hosts = set(r["hostname"] for r in results)
print(f"Tasks distributed across {len(unique_hosts)} nodes:")
for host in sorted(unique_hosts):
    count = sum(1 for r in results if r["hostname"] == host)
    print(f"  {host}: {count} tasks")
'''

print("Verifying Multi-Node Distribution")
print("=" * 50)
print(verification_code)

Verifying Multi-Node Distribution

@ray.remote
def get_node_info():
    """Return information about the node running this task."""
    import socket
    import os
    return {
        "hostname": socket.gethostname(),
        "pid": os.getpid(),
        "ray_node_id": ray.get_runtime_context().get_node_id(),
    }

# Run on multiple workers
n_tasks = 100
futures = [get_node_info.remote() for _ in range(n_tasks)]
results = ray.get(futures)

# Count unique nodes
unique_hosts = set(r["hostname"] for r in results)
print(f"Tasks distributed across {len(unique_hosts)} nodes:")
for host in sorted(unique_hosts):
    count = sum(1 for r in results if r["hostname"] == host)
    print(f"  {host}: {count} tasks")



---

## 4. Local Simulation

Let's simulate multi-node behavior locally:

In [ ]:
# Initialize Ray locally (simulating a small cluster)
if ray.is_initialized():
    ray.shutdown()

# Use 4 CPUs to simulate 4 "workers"
ray.init(num_cpus=4, logging_level="WARNING")

print(f"Local Ray initialized")
print(f"Resources: {ray.cluster_resources()}")

Local Ray initialized
Resources: {'CPU': 4.0, 'object_store_memory': 3989336064.0, 'node:172.28.0.12': 1.0, 'node:__internal_head__': 1.0, 'memory': 9308450816.0}


/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


In [ ]:
# Simulate distributed processing
import time

@ray.remote
def process_partition(partition_id, data_size):
    """Simulate processing a data partition."""
    import socket

    # Simulate work
    data = np.random.randn(data_size)
    result = np.mean(data ** 2)

    return {
        "partition_id": partition_id,
        "result": result,
        "worker_pid": os.getpid(),
    }

# Simulate processing 16 partitions across 4 workers
n_partitions = 16
partition_size = 100_000

print(f"Processing {n_partitions} partitions...")
start = time.time()

futures = [process_partition.remote(i, partition_size) for i in range(n_partitions)]
results = ray.get(futures)

elapsed = time.time() - start

# Analyze distribution
workers = set(r["worker_pid"] for r in results)
print(f"\nCompleted in {elapsed:.2f}s")
print(f"Partitions processed by {len(workers)} workers")

for worker in sorted(workers):
    partitions = [r["partition_id"] for r in results if r["worker_pid"] == worker]
    print(f"  Worker {worker}: partitions {partitions}")

Processing 16 partitions...

Completed in 4.97s
Partitions processed by 4 workers
  Worker 2051: partitions [1, 4, 7, 9, 11, 12, 15]
  Worker 2052: partitions [0, 5, 6, 8, 10, 13, 14]
  Worker 2139: partitions [3]
  Worker 2140: partitions [2]


---

## 5. Scaling Considerations

In [ ]:
# Scaling efficiency analysis

@ray.remote
def compute_intensive_task(n):
    """CPU-bound task for scaling test."""
    data = np.random.randn(n)
    return np.sum(np.sin(data) ** 2 + np.cos(data) ** 2)

def measure_scaling(n_tasks, task_size):
    """Measure execution time with given number of tasks."""
    start = time.time()
    futures = [compute_intensive_task.remote(task_size) for _ in range(n_tasks)]
    results = ray.get(futures)
    return time.time() - start

# Test with increasing task counts
task_size = 1_000_000
task_counts = [1, 2, 4, 8, 16]

print("Scaling Test Results")
print("=" * 50)
print(f"{'Tasks':>8} {'Time (s)':>12} {'Speedup':>10} {'Efficiency':>12}")
print("-" * 50)

baseline_time = None
for n_tasks in task_counts:
    elapsed = measure_scaling(n_tasks, task_size)

    if baseline_time is None:
        baseline_time = elapsed
        speedup = 1.0
    else:
        speedup = baseline_time * n_tasks / elapsed

    efficiency = speedup / min(n_tasks, 4) * 100  # 4 CPUs available

    print(f"{n_tasks:>8} {elapsed:>12.3f} {speedup:>10.2f}x {efficiency:>11.1f}%")

Scaling Test Results
   Tasks     Time (s)    Speedup   Efficiency
--------------------------------------------------
       1        0.134       1.00x       100.0%
       2        0.251       1.07x        53.5%
       4        0.516       1.04x        26.0%
       8        0.957       1.12x        28.1%
      16        2.058       1.04x        26.1%


### Memory Management Tips

In [ ]:
memory_tips = '''
Memory Management for Multi-Node Ray
====================================

1. OBJECT STORE SIZING
   - Default: 30% of system memory
   - Increase for data-heavy workloads:
     ray.init(object_store_memory=100 * 1024**3)  # 100 GB

2. ENABLE SPILLING
   - Spill objects to disk when object store is full:
     ray.init(
         _system_config={
             "object_spilling_config": {
                 "type": "filesystem",
                 "params": {
                     "directory_path": "/scratch/$USER/ray_spill"
                 }
             }
         }
     )

3. USE RAY.PUT() FOR SHARED DATA
   - Put large objects once, reference many times:
     large_data_ref = ray.put(large_data)
     futures = [task.remote(large_data_ref) for _ in range(100)]

4. BATCH RESULTS
   - Use ray.wait() instead of ray.get() for streaming:
     while futures:
         done, futures = ray.wait(futures, num_returns=1)
         process(ray.get(done[0]))

5. MONITOR MEMORY
   - Check dashboard: http://head-node:8265
   - Use: ray memory
'''

print(memory_tips)


Memory Management for Multi-Node Ray

1. OBJECT STORE SIZING
   - Default: 30% of system memory
   - Increase for data-heavy workloads:
     ray.init(object_store_memory=100 * 1024**3)  # 100 GB

2. ENABLE SPILLING
   - Spill objects to disk when object store is full:
     ray.init(
         _system_config={
             "object_spilling_config": {
                 "type": "filesystem",
                 "params": {
                     "directory_path": "/scratch/$USER/ray_spill"
                 }
             }
         }
     )

3. USE RAY.PUT() FOR SHARED DATA
   - Put large objects once, reference many times:
     large_data_ref = ray.put(large_data)
     futures = [task.remote(large_data_ref) for _ in range(100)]

4. BATCH RESULTS
   - Use ray.wait() instead of ray.get() for streaming:
     while futures:
         done, futures = ray.wait(futures, num_returns=1)
         process(ray.get(done[0]))

5. MONITOR MEMORY
   - Check dashboard: http://head-node:8265
   - Use: ray memory

---

## 6. Exercise: Design a Multi-Node Job

In [ ]:
# Exercise: Calculate resources for this scenario

scenario = """
SCENARIO:
=========
You need to train an XGBoost model on:
- 300 GB Parquet dataset
- 50 features
- Regression task

Requirements:
- Entire dataset should fit in cluster memory
- Training should complete in < 2 hours
- Need 2x data size for processing overhead

QUESTIONS:
1. How many Expanse nodes do you need?
2. What partition should you use?
3. How many Ray workers should you configure?
4. What object store size per node?
"""

print(scenario)


SCENARIO:
You need to train an XGBoost model on:
- 300 GB Parquet dataset
- 50 features
- Regression task

Requirements:
- Entire dataset should fit in cluster memory
- Training should complete in < 2 hours
- Need 2x data size for processing overhead

QUESTIONS:
1. How many Expanse nodes do you need?
2. What partition should you use?
3. How many Ray workers should you configure?
4. What object store size per node?



In [ ]:
# Solution

solution = """
SOLUTION:
=========

1. NODES NEEDED:
   - Data size: 300 GB
   - With 2x overhead: 600 GB needed
   - Per node: 256 GB RAM, ~180 GB usable for Ray
   - Nodes needed: 600 / 180 = 3.3 → 4 nodes

2. PARTITION:
   - Need multiple nodes → compute partition
   - #SBATCH --partition=compute

3. RAY WORKERS:
   - 4 nodes × 64 cores allocated = 256 cores
   - XGBoost works well with 8 cores per worker
   - num_workers = 256 / 8 = 32

4. OBJECT STORE:
   - Request 200G memory per node
   - Object store: ~100 GB per node
   - Total: 400 GB object store

SLURM CONFIGURATION:
   #SBATCH --nodes=4
   #SBATCH --cpus-per-task=64
   #SBATCH --mem=200G
   #SBATCH --partition=compute
   #SBATCH --time=02:00:00

RAY TRAIN CONFIGURATION:
   ScalingConfig(
       num_workers=32,
       resources_per_worker={"CPU": 8},
   )
"""

print(solution)


SOLUTION:

1. NODES NEEDED:
   - Data size: 300 GB
   - With 2x overhead: 600 GB needed
   - Per node: 256 GB RAM, ~180 GB usable for Ray
   - Nodes needed: 600 / 180 = 3.3 → 4 nodes

2. PARTITION:
   - Need multiple nodes → compute partition
   - #SBATCH --partition=compute

3. RAY WORKERS:
   - 4 nodes × 64 cores allocated = 256 cores
   - XGBoost works well with 8 cores per worker
   - num_workers = 256 / 8 = 32

4. OBJECT STORE:
   - Request 200G memory per node
   - Object store: ~100 GB per node
   - Total: 400 GB object store

SLURM CONFIGURATION:
   #SBATCH --nodes=4
   #SBATCH --cpus-per-task=64
   #SBATCH --mem=200G
   #SBATCH --partition=compute
   #SBATCH --time=02:00:00

RAY TRAIN CONFIGURATION:
   ScalingConfig(
       num_workers=32,
       resources_per_worker={"CPU": 8},
   )



---

## Summary

### Multi-Node Ray on SLURM

1. **Architecture**: Head node + worker nodes across SLURM allocation
2. **Connection**: `ray.init(address='head-ip:6379')`
3. **Scaling**: More nodes = more memory and cores
4. **Memory**: Configure object store, enable spilling

### SLURM Script Key Elements

```bash
#SBATCH --nodes=N           # Number of nodes
#SBATCH --partition=compute # For multi-node

# Start head node
ray start --head --port=6379

# Start workers
ray start --address=HEAD_IP:6379
```

### Best Practices

1. Start small (1-2 nodes), scale up as needed
2. Use debug partition for testing
3. Monitor with Ray dashboard
4. Enable object spilling for large datasets

In [ ]:
# Cleanup
ray.shutdown()
print("Ray shutdown complete.")

Ray shutdown complete.


---

## Course Summary

### What We Covered in Class16

| Module | Topic | Key Concepts |
|--------|-------|-------------|
| 1 | Parallelization & HPC | Amdahl's Law, SDSC Expanse |
| 2 | SLURM & Spark | Job scripts, partitions, containers |
| 3 | Introduction to Ray | Tasks, Actors, Ray Data, Ray Train |
| 4 | Ray + Spark Integration | RayDP, data handoff patterns |
| 5 | Ray on SLURM | Multi-node clusters, scaling |

### What You Can Do Now

1. Run Spark and Ray jobs on SDSC Expanse
2. Choose the right framework for your workload
3. Scale ML training beyond single-machine limits
4. Integrate Spark ETL with Ray ML pipelines